### LoRA vs. QLoRA: A Comparison (Table Format)

| **Aspect**               | **LoRA**                                                                                                      | **QLoRA**                                                                                                     |
|---------------------------|----------------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------------|
| **Core Concept**          | Parameter-efficient fine-tuning using trainable low-rank matrices (A and B).                                  | Combines LoRA with quantization (4-bit NormalFloat) for reduced memory usage.                                 |
| **Memory Usage**          | Requires ~14–18 GB VRAM for a 7B model.                                                                       | Reduces peak GPU memory by up to 75% (~9.1 GB VRAM for a 7B model).                                           |
| **Training Speed**        | Faster training process due to fewer computational steps.                                                     | ~30% slower than LoRA due to added quantization and memory management overhead.                              |
| **Model Parameters**      | Original weights remain frozen; only low-rank matrices (A, B) are updated.                                   | Similar approach: base model is frozen, and only adapters are updated.                                       |
| **Adaptability**          | Suitable for tasks requiring a moderate memory footprint.                                                     | Ideal for environments with strict GPU memory constraints.                                                   |
| **Quantization**          | No quantization involved.                                                                                     | Uses a 4-bit NF4 data type and double quantization for efficient memory usage.                               |
| **Application**           | Fine-tunes efficiently with fewer resources and faster results.                                               | Enables fine-tuning large models on hardware with limited GPU memory.                                        |
| **Performance Trade-offs**| Comparable to full fine-tuning in quality.                                                                   | Similar quality but slightly slower due to memory optimizations.                                             |
| **Flexibility**           | Supports multiple LoRA weight combinations for task/domain switching.                                         | Provides memory savings at the cost of slower training speed.                                                |

### Key Metrics to Monitor During Training

![](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781836200079/files/Images/B31105_05_11.png)

#### **1. Training Loss**
- **Definition**: Measures how well the model is learning the task during training.
- **Behavior**: 
  - Should decrease over time, typically showing rapid initial drops followed by a plateau.
  - Spikes or continuous increases indicate issues such as poor data quality or suboptimal parameters.
- **Actions**:
  - Inspect data quality and tokenizer issues.
  - Adjust hyperparameters like learning rate or batch size.

---

#### **2. Validation Loss**
- **Definition**: Measures model performance on unseen validation data.
- **Behavior**:
  - Should decrease alongside training loss and stabilize with a small gap.
  - Increasing validation loss alongside decreasing training loss signals overfitting.
  - Both remaining high may indicate underfitting.
- **Actions**:
  - Monitor convergence and stability for both curves.
  - Address overfitting by techniques like regularization.

---

#### **3. Gradient Norm**
- **Definition**: Reflects the magnitude of the gradient during training.
- **Behavior**:
  - Large values can signify instability (potential overfitting).
  - Stable or decreasing norms suggest convergence.
- **Actions**:
  - Apply gradient clipping to cap the gradient norm and stabilize updates.

Monitoring these metrics ensures a balanced and effective training process. Let me know if you'd like to explore methods for addressing specific issues like overfitting or underfitting!

### Biases in LLM-Based Evaluation

#### **1. Position Bias**
- **Description**: LLM judges often favor the first answer presented during relative scoring.
- **Impact**: Can lead to skewed evaluations and inaccurate preference rankings.

---

#### **2. Length Bias**
- **Description**: Preference for longer answers, similar to human tendencies.
- **Impact**: Risks undervaluing concise responses, even if they are of higher quality.

---

#### **3. Family Bias**
- **Description**: LLM judges may favor responses generated by themselves or models of the same family.
- **Impact**: Occurs due to shared patterns or knowledge bases, leading to biased evaluations.

### Strategies to Mitigate Biases in Preference Datasets

#### **1. Randomizing the Order of Responses**
- **What It Does**: Alternates the position of answer A and answer B in comparisons.
- **Impact**: Reduces **position bias** by ensuring evaluation doesn’t favor the first response.

---

#### **2. Using Few-Shot Examples**
- **What It Does**: Provides examples with balanced score distributions (e.g., shorter vs. longer answers, or responses from diverse model families).
- **Impact**:
  - Calibrates LLM judges to avoid **length bias**.
  - Demonstrates that concise or cross-family responses can also be of high quality.

---

#### **3. Employing Multiple Models (Jury Approach)**
- **What It Does**: Uses multiple models to evaluate responses instead of relying on a single LLM judge.
- **Impact**:
  - Balances individual biases inherent to any single model.
  - Enhances evaluation robustness by offering diverse perspectives.





| **Category**               | **Benchmarks/Evaluations**           | **What They Measure**                                                                              | **Limitations**                                                                                                  |
|----------------------------|--------------------------------------|----------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------|
| **General Knowledge**      | MMLU, TruthfulQA                     | Broad factual knowledge across diverse subjects and handling common misconceptions.                | May lack depth for specialized domains and miss evolving misinformation nuances.                                |
| **Reasoning**              | BBH, GSM8K                           | Logical planning, complex multi-step reasoning, and mathematical problem-solving.                  | Often use artificial puzzles that might not fully capture real-world ambiguity.                                  |
| **Language Understanding** | HELM, WinoGrande                     | Overall language processing from fluency to contextual understanding and pronoun disambiguation.   | Aggregated metrics can mask issues in domain-specific or dynamic conversational contexts.                         |
| **LLM-Specific Evaluations** | MT-Bench, AlpacaEval               | Comprehensive performance evaluation on alignment, helpfulness, and overall response quality.       | Aggregate scores may overlook fine-grained performance differences and subtle challenges seen in real-world tasks.  |


\


# Perplexity f

## Understanding Perplexity (PPL)
## **1. What Is Perplexity?**

Perplexity is a standard metric for evaluating language models. It quantifies how “surprised” a model is when predicting the next token in a sequence. 
-  perplexity is defined as the exponentiation of the cross-entropy loss:

$$ \text{Perplexity} = e^{\text{Cross-Entropy Loss}}\ $$

- **Interpretation:**

- **Lower perplexity** implies the model is better at predicting the sequence (i.e., it is less "perplexed").

- **Higher perplexity** indicates more uncertainty and poorer predictive performance.

## **2. Why Use Perplexity?**

-   **Performance Indicator:** It provides a single numerical value that summarizes the quality of probabilistic predictions made by the model.
    
-   **Comparison Across Models:** Since it is based on the language model’s loss, perplexity can help compare different models or fine-tuned versions on the same corpus.
    
-   **Smooths Over Sequence Length:** By taking the exponentiation of the loss, perplexity naturally scales with longer sequences in a way that is easier to interpret.
### Challenge with Fixed-Length Models

Models like GPT-2 have a fixed context window (e.g., 1024 tokens). When evaluating PPL, we need to handle sequences longer than this limit efficiently.

## Methods to Calculate Perplexity

### 1. **Naïve Approach: Disjoint Chunks** (Suboptimal)
![Suboptimal PPL not taking advantage of full available context](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/ppl_chunked.gif)
-   Splits the sequence into independent chunks.
-   Computes perplexity for each segment separately.
-   **Drawback**: Ignores contextual dependencies, leading to inflated PPL.

### 2. **Better Approach: Sliding Window Strategy**
![Sliding window PPL taking advantage of all available context](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/ppl_sliding.gif)
-   Uses overlapping context windows.
-   Ensures better context utilization.

## Implementation with Code

### Step 1: Load Model & Tokenizer

```python
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

device = 'cuda'  # or 'cpu'
model_id = 'gpt2-large'
model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
tokenizer = GPT2TokenizerFast.from_pretrained(model_id)

```

### Step 2: Tokenize Input Text

```python
text = "Your input text goes here."
encodings = tokenizer(text, return_tensors='pt')

```

### Step 3: Sliding Window Parameters

```python
max_length = model.config.n_positions  # Maximum context size
stride = 512  # Overlap between windows

```

### Step 4: Compute Perplexity Using Sliding Window

```python
import torch
import math

lls = []
for i in range(0, encodings.input_ids.size(1), stride):
    begin_loc = max(i + stride - max_length, 0)
    end_loc = min(i + stride, encodings.input_ids.size(1))
    trg_len = end_loc - i  # Tokens to predict
    input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
    target_ids = input_ids.clone()
    target_ids[:, :-trg_len] = -100  # Mask context tokens
    
    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)
        log_likelihood = outputs.loss * trg_len
    lls.append(log_likelihood)

ppl = torch.exp(torch.stack(lls).sum() / end_loc)

6. Beyond Perplexity
While perplexity is a powerful measure, it is one of many evaluation metrics. Depending on your use case, you might also consider:

```
## **6. Beyond Perplexity**

While perplexity is a powerful measure, it is one of many evaluation metrics. Depending on your use case, you might also consider:

-   **BLEU Score:** For translation tasks.
    
-   **ROUGE Score:** For summarization tasks.
    
-   **Human Evaluation:** To capture qualitative aspects that perplexity might miss.






After pre-training, it is common to use a suite of evaluations to evaluate the base model. This suite can include internal and public benchmarks. Here’s a non-exhaustive list of common public pre-training evaluations:

-   **MMLU (knowledge)**: Tests models on multiple-choice questions across 57 subjects, from elementary to professional levels
-   **HellaSwag (reasoning)**: Challenges models to complete a given situation with the most plausible ending from multiple choices
-   **ARC-C (reasoning)**: Evaluates models on grade-school-level multiple-choice science questions requiring causal reasoning.


fine-tuned models also have their own benchmarks. Here, we use the term “fine-tuned model” to designate a model that has been trained  with  **supervised fine-tuning**  (**SFT**) 

| **Evaluation**   | **Category**          | **Description**                                                                                                                                             | **Key Capabilities Tested**                                                |
|------------------|-----------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------|
| **IFEval**       | Instruction Following | Assesses a model’s ability to follow instructions under specific constraints (e.g., not using any commas in the output).                                     | Instruction adherence, constraint handling                               |
| **Chatbot Arena**| Conversation          | A human-centric framework where evaluators vote for the best answer produced by two models in a head-to-head conversation scenario.                        | Conversational quality, context sensitivity, human judgment comparison     |
| **AlpacaEval**   | Instruction Following | An automatic evaluation designed for fine-tuned models that shows high correlation with the human judgments obtained from Chatbot Arena.                 | Instruction following consistency, automatic performance correlation       |
| **MT-Bench**     | Conversation          | Evaluates models in multi-turn conversations, focusing on their ability to retain context and deliver coherent, sustained responses across turns.         | Multi-turn context management, coherence, dialogue consistency             |
| **GAIA**         | Agentic               | Tests a wide range of advanced abilities—including tool use, web browsing, and multi-step reasoning—by putting models in more complex, task-driven scenarios. | Agentic behavior, multi-step reasoning, integration of external tools      |


>Recommend lm-evaluation-harness ([github.com/EleutherAI/lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)) from Eleuther AI and lighteval ([github.com/huggingface/lighteval](https://github.com/huggingface/lighteval)) from Hugging Face to run your benchmarks.




LLM as a Judge: If you have ground-truth answers, providing them as additional context improves  the accuracy of the evaluation :

```
You are an evaluator who assesses the quality of an answer to an instruction.

Your goal is to provide a score that represents how well the answer addresses the instruction.

You will use a scale of 1 to 4, where each number represents the following:

1. The answer is not relevant to the instruction.

2. The answer is relevant but not helpful.

3. The answer is relevant and helpful but could be more detailed.

4. The answer is relevant, helpful, and detailed.

Please provide your evaluation as follows:

##Evaluation##

Explanation: (analyze the relevant, helpfulness, and complexity of the answer)

Total rating: (final score as a number between 1 and 4)

**Instruction**:

{instruction}

**Answer**:

{answer}

##Evaluation##

Explanation:
```

The evaluation of RAG systems goes beyond assessing a standalone LLM. It requires examining the entire system’s performance, including:

-   **Retrieval accuracy**: How well does the system fetch relevant information?
-   **Integration quality**: How effectively is the retrieved information incorporated into the generated response?
-   **Factuality and relevance**: Does the final output address the query appropriately while seamlessly blending retrieved and generated content?
REF: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/#example
- 
![](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781836200079/files/Images/B31105_07_01.png)


### Relationships & Metrics

1.  **Answer Relevance** (From _Question_ to _Answer_)
    
    -   **Definition**: How well the answer addresses the user’s actual question.
    -   **Why It Matters**: Even if the answer is factually correct about some topic, it may be irrelevant if it doesn’t actually address the question.
2.  **Faithfulness** (From _Answer_ to _Context_)
    
    -   **Definition**: How accurately the answer reflects (or remains faithful to) the given context.
    -   **Why It Matters**: Ensures that the answer isn’t just “made up” (hallucinated) but actually grounded in the provided source information.
3.  **Context Precision** (From _Question_ to _Context_)
    
    -   **Definition**: How relevant or precise the chosen context is with respect to the question.
    -   **Why It Matters**: The system should retrieve or focus on the parts of the context that actually answer the question, rather than retrieving irrelevant or overly broad text.
4.  **Context Recall** (From _Context_ to _Ground Truth_)
    
    -   **Definition**: How well the context covers all necessary information that appears in the human‐curated ground truth.
    -   **Why It Matters**: If the context is missing key information, the answer might be incomplete or inaccurate..

### 1. Context Precision

**Definition:** Context Precision measures the `proportion of retrieved context chunks that are relevant to the question`. It assesses the accuracy of the retrieval component in selecting pertinent information.​

**Evaluation Methods:**

-   **Precision@k:** Calculate the ratio of relevant chunks among the top-k retrieved contexts.
    
    -   Retrieving the top-k context chunks based on the system's ranking.​
    -   Determining the number of these chunks that are relevant to the question.​
        
    -   Computing Precision@k as:​
         $$ \text{Precision@k} = \frac{\text{Number of Relevant Chunks in Top-k}}{k} $$
         Precision@k=kNumber of Relevant Chunks in Top-k​
        
    
    For example, if 3 out of the top 5 retrieved chunks are relevant, Precision@5 = 0.6.​
    
-   **Average Precision (AP):** Calculate the average of Precision@k values at each position k where a relevant chunk is retrieved. This provides a single metric that considers the ranking order of relevant chunks.​
    

**Tools and Frameworks:**

-   **Ragas:** A framework that offers metrics like Context Precision to evaluate the proportion of relevant chunks in retrieved contexts.

### 2. Context Recall

**Definition:** Context Recall measures the proportion of relevant information from the ground truth that is present in the retrieved contexts. It evaluates the system's ability to capture all necessary information to answer the question accurately.​

**Evaluation Methods:**

-   **Recall@k:** Calculate the ratio of relevant information retrieved among the top-k contexts to the total relevant information available.
    -   Identifying all relevant information (ground truth) needed to answer the question.​
    -   Determining how much of this information is present in the top-k retrieved contexts.​
        
    -   Computing Recall@k as:​
        
        Recall@k=Amount of Relevant Information in Top-kTotal Relevant Information Available
        - $$ \text{Recall@k} = \frac{\text{Amount of Relevant Information in Top-k}}{\text{Total Relevant Information Available}} $$
        - Recall@k=Total Relevant Information AvailableAmount of Relevant Information in Top-k​
        
    
    For instance, if the ground truth contains 10 relevant facts and the top 5 retrieved contexts contain 7 of these facts, Recall@5 = 0.7.



LLM as a Judge: If you have ground-truth answers, providing them as additional context improves  the accuracy of the evaluation :

```
You are an evaluator who assesses the quality of an answer to an instruction.

Your goal is to provide a score that represents how well the answer addresses the instruction.

You will use a scale of 1 to 4, where each number represents the following:

1. The answer is not relevant to the instruction.

2. The answer is relevant but not helpful.

3. The answer is relevant and helpful but could be more detailed.

4. The answer is relevant, helpful, and detailed.

Please provide your evaluation as follows:

##Evaluation##

Explanation: (analyze the relevant, helpfulness, and complexity of the answer)

Total rating: (final score as a number between 1 and 4)

**Instruction**:

{instruction}

**Answer**:

{answer}

##Evaluation##

Explanation:
```

The evaluation of RAG systems goes beyond assessing a standalone LLM. It requires examining the entire system’s performance, including:

-   **Retrieval accuracy**: How well does the system fetch relevant information?
-   **Integration quality**: How effectively is the retrieved information incorporated into the generated response?
-   **Factuality and relevance**: Does the final output address the query appropriately while seamlessly blending retrieved and generated content?
REF: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/#example
- 
![](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781836200079/files/Images/B31105_07_01.png)


### Relationships & Metrics

1.  **Answer Relevance** (From _Question_ to _Answer_)
    
    -   **Definition**: How well the answer addresses the user’s actual question.
    -   **Why It Matters**: Even if the answer is factually correct about some topic, it may be irrelevant if it doesn’t actually address the question.
2.  **Faithfulness** (From _Answer_ to _Context_)
    
    -   **Definition**: How accurately the answer reflects (or remains faithful to) the given context.
    -   **Why It Matters**: Ensures that the answer isn’t just “made up” (hallucinated) but actually grounded in the provided source information.
3.  **Context Precision** (From _Question_ to _Context_)
    
    -   **Definition**: How relevant or precise the chosen context is with respect to the question.
    -   **Why It Matters**: The system should retrieve or focus on the parts of the context that actually answer the question, rather than retrieving irrelevant or overly broad text.
4.  **Context Recall** (From _Context_ to _Ground Truth_)
    
    -   **Definition**: How well the context covers all necessary information that appears in the human‐curated ground truth.
    -   **Why It Matters**: If the context is missing key information, the answer might be incomplete or inaccurate..

### 1. Context Precision

**Definition:** Context Precision measures the `proportion of retrieved context chunks that are relevant to the question`. It assesses the accuracy of the retrieval component in selecting pertinent information.​

**Evaluation Methods:**

-   **Precision@k:** Calculate the ratio of relevant chunks among the top-k retrieved contexts.
    
    -   Retrieving the top-k context chunks based on the system's ranking.​
    -   Determining the number of these chunks that are relevant to the question.​
        
    -   Computing Precision@k as:​
         $$ \text{Precision@k} = \frac{\text{Number of Relevant Chunks in Top-k}}{k} $$
         Precision@k=kNumber of Relevant Chunks in Top-k​
        
    
    For example, if 3 out of the top 5 retrieved chunks are relevant, Precision@5 = 0.6.​
    
-   **Average Precision (AP):** Calculate the average of Precision@k values at each position k where a relevant chunk is retrieved. This provides a single metric that considers the ranking order of relevant chunks.​
    

**Tools and Frameworks:**

-   **Ragas:** A framework that offers metrics like Context Precision to evaluate the proportion of relevant chunks in retrieved contexts.

### 2. Context Recall

**Definition:** Context Recall measures the proportion of relevant information from the ground truth that is present in the retrieved contexts. It evaluates the system's ability to capture all necessary information to answer the question accurately.​

**Evaluation Methods:**

-   **Recall@k:** Calculate the ratio of relevant information retrieved among the top-k contexts to the total relevant information available.
    -   Identifying all relevant information (ground truth) needed to answer the question.​
    -   Determining how much of this information is present in the top-k retrieved contexts.​
        
    -   Computing Recall@k as:​
        
        Recall@k=Amount of Relevant Information in Top-kTotal Relevant Information Available
        - $$ \text{Recall@k} = \frac{\text{Amount of Relevant Information in Top-k}}{\text{Total Relevant Information Available}} $$
        - Recall@k=Total Relevant Information AvailableAmount of Relevant Information in Top-k​
        
    
    For instance, if the ground truth contains 10 relevant facts and the top 5 retrieved contexts contain 7 of these facts, Recall@5 = 0.7.

-   **ROUGE (Recall-Oriented Understudy for Gisting Evaluation):**
    
    -   **Definition:** ROUGE measures the overlap between the retrieved text and the ground truth by evaluating the `n-gram`, `word sequence`, and` word pair matches`.
    -  It focuses on recall, indicating how much of the ground truth is captured by the retrieved content.​
    -   **Application:** By comparing the retrieved text to the ground truth, ROUGE provides insight into the completeness of the retrieval process.​

-   **BERTScore:**
    
    -   **Definition:** BERTScore leverages `contextual embeddings` from BERT to compute similarity scores between the `retrieved text` and the `ground truth` on a token-by-token basis. This method captures semantic similarities beyond exact matches.​
    -   **Application:** BERTScore assesses the `semantic alignment` between the retrieved text and the ground truth, offering a nuanced evaluation of retrieval quality.

-   **GLEU (Google-BLEU) Score:**
    
    -   **Definition:** GLEU measures the `similarity between the retrieved text and the ground truth` by evaluating `shared n-grams`, focusing on both precision and recall. **It addresses some limitations of the traditional BLEU score by considering sentence-level variations**.​
    -   **Application:** GLEU provides a more flexible evaluation of retrieval quality, accommodating variations in phrasing while maintaining alignment with the ground truth.​
    
-   **METEOR (Metric for Evaluation of Translation with Explicit ORdering):**
    
    -   **Definition:** METEOR evaluates the similarity between the retrieved text and the ground truth by considering `synonyms`, `stemming`, and `paraphrasing`, in addition to `exact n-gram matches`.​
    -   **Application:** METEOR offers a comprehensive assessment of retrieval quality by accounting for `linguistic variations and semantic equivalence`.


The `safetensors` format is used to store model weights and other tensors securely and efficiently. Here’s why we use it instead of traditional formats like `.pt` or `.bin`:

### **Reasons for Using `safetensors`**

1.  **Security (Avoid Arbitrary Code Execution) 🔒**
    
    -   Traditional formats like `.pt` (PyTorch) and `.bin` (Pickle-based) can execute arbitrary Python code during loading, making them vulnerable to malicious modifications.
    -   `safetensors` is designed to **only** store tensor data without any executable code, making it much safer.
2.  **Speed (Memory-Mapped Loading) ⚡**
    
    -   `safetensors` uses memory-mapped files, allowing models to be loaded **without duplicating memory usage**.
    -   This speeds up inference and training, especially for large models.
3.  **Efficient Storage & Compatibility 📦**
    
    -   It supports efficient serialization of large tensor data while keeping them aligned for fast access.
    -   Compatible with PyTorch, TensorFlow, and JAX.
4.  **Reliability & Integrity ✅**
    
    -   The format ensures **consistent tensor sizes and types** to prevent accidental corruption.
    -   Unlike `pickle`, which can sometimes break across different Python versions, `safetensors` remains stable.

### **Example Usage in PyTorch**

```python
from safetensors.torch import save_file

state_dict = model.state_dict()  # Get model weights
save_file(state_dict, "model.safetensors")  # Save in safetensors format

```

### **Loading the Model**

```python
from safetensors.torch import load_file

state_dict = load_file("model.safetensors")  
model.load_state_dict(state_dict)

```